# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kuteesatendojeremiah/Tendojerry-Flyrank/blob/main/work/notebooks/w04_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip install -q duckdb huggingface_hub

import os, getpass
import duckdb
import pandas as pd

# CI and power users set HF_TOKEN in the environment or Colab Secrets (🔑 icon);
# everyone else gets the safe interactive prompt. Never hardcode the token in a cell — this repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Same dev window as the data contract (w03) — mid-panel month, never the sealed June 2026 sample.
MONTH_START = "2026-03-01"
MONTH_END_EXCL = "2026-04-01"      # half-open: report_date < MONTH_END_EXCL
PREV30_START = "2026-01-30"        # the 30 days immediately before MONTH_START
PREV30_END_EXCL = MONTH_START

print(f"Connected. Iterating on month={MONTH_START[:7]} | prev30 window: [{PREV30_START}, {PREV30_END_EXCL})")


Paste your Hugging Face READ token (hf_...): ··········
Connected. Iterating on month=2026-03 | prev30 window: [2026-01-30, 2026-03-01)


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
# --- Numeric prev30 features (same 5 as the data contract, w03) ---
# avg_position_prev30 uses NULLIF(gsc_avg_position, 0): per docs/data-dictionary.md, 0 in this
# field means "no position data that day", not literal position zero — averaging it in unfiltered
# would silently drag every item's average toward zero.
feature_frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_prev30,
           SUM(gsc_clicks) AS clk_prev30,
           AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_prev30,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days_prev30,
           COUNT(*) AS rows_prev30,
           SUM(CASE WHEN gsc_avg_position = 0 THEN 1 ELSE 0 END) AS zero_position_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{PREV30_START}' AND report_date < DATE '{PREV30_END_EXCL}'
    GROUP BY 1, 2
""").df()

pct_zero_position = feature_frame["zero_position_rows"].sum() / feature_frame["rows_prev30"].sum() * 100
print(f"gsc_avg_position == 0 (no position data) in {pct_zero_position:.1f}% of prev30 rows — excluded from the average above.")

# --- Fills ---
# imp_prev30 == 0 => clk_prev30 is also 0 => ctr_prev30 would be 0/0 (NaN). That's a real
# "no opportunity yet" state, not a missing measurement, so it's filled with 0 rather than dropped.
feature_frame["ctr_prev30"] = (feature_frame["clk_prev30"] / feature_frame["imp_prev30"]).fillna(0)

# avg_position_prev30 can still be NaN for an item whose every prev30 row had gsc_avg_position == 0
# (active but never ranked with a real position) — kept as NaN and reported, not silently
# zero-filled (a 0 there would misread as "position zero", the best possible rank).
n_unranked = feature_frame["avg_position_prev30"].isna().sum()
print(f"{n_unranked:,} of {len(feature_frame):,} prev30 items have no real position data at all (kept as NaN, not filled).")

feature_frame = feature_frame.drop(columns=["rows_prev30", "zero_position_rows"])
print(f"Numeric feature frame: {len(feature_frame):,} content items x {feature_frame.shape[1] - 2} numeric features")
feature_frame.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

gsc_avg_position == 0 (no position data) in 2.4% of prev30 rows — excluded from the average above.
168,485 of 321,546 prev30 items have no real position data at all (kept as NaN, not filled).
Numeric feature frame: 321,546 content items x 5 numeric features


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,avg_position_prev30,active_days_prev30,ctr_prev30
0,client_3ffa76342f366962,content_5067dafdab76f97c,0.0,0.0,NaN,0,0.0
1,client_3ffa76342f366962,content_cfd5f8764543bfee,1.0,0.0,5.0,1,0.0
2,client_3ffa76342f366962,content_6d26e2f1a8a6512b,0.0,0.0,NaN,0,0.0
3,client_3ffa76342f366962,content_ab514679d7b70992,0.0,0.0,NaN,0,0.0
4,client_3ffa76342f366962,content_f1a36b8d0b297a35,0.0,0.0,NaN,0,0.0


In [3]:
# --- Categorical handling: discover dim_content's schema live ---
# dim_content's exact columns aren't confirmed anywhere in this repo's docs/notebooks (only its
# row count is, from w03) — DESCRIBE first rather than guess a column name this run might not have.
content_schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()
print(content_schema)


                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

In [4]:
# Pick safe, low-cardinality text columns to one-hot encode: VARCHAR type, 2-15 distinct values,
# and not an obvious identifier/account/product-metadata column (those are handled in Section 4,
# not used here even provisionally — a flag or score describes a decision, not observed content).
EXCLUDE_LIKE = ("client", "hash", "id", "profile", "account", "flag", "score")
candidate_cols = [
    c for c in content_schema.loc[content_schema["column_type"] == "VARCHAR", "column_name"]
    if not any(bad in c.lower() for bad in EXCLUDE_LIKE)
]

cat_features = []
for col in candidate_cols:
    n_distinct = con.sql(f"SELECT COUNT(DISTINCT {col}) FROM {TABLES['dim_content']}").fetchone()[0]
    if 1 < n_distinct <= 15:
        cat_features.append(col)

print(f"Categorical candidates found on dim_content: {cat_features}")

if cat_features:
    cols_sql = ", ".join(cat_features)
    content_meta = con.sql(f"""
        SELECT content_hash_id, {cols_sql}
        FROM {TABLES['dim_content']}
    """).df()

    feature_frame = feature_frame.merge(content_meta, on="content_hash_id", how="left")

    # Missing category -> its own explicit "unknown" level. A blind fillna into the reference
    # category would silently fold "no metadata" into whichever category get_dummies drops first.
    for col in cat_features:
        n_missing = feature_frame[col].isna().sum()
        print(f"{col}: {n_missing:,} missing ({n_missing / len(feature_frame) * 100:.1f}%) -> filled 'unknown'")
        feature_frame[col] = feature_frame[col].fillna("unknown")

    feature_frame = pd.get_dummies(feature_frame, columns=cat_features, prefix=cat_features)
    print(f"Feature frame with categoricals: {len(feature_frame):,} rows x {feature_frame.shape[1]} columns")
else:
    print("No safe low-cardinality categorical column found on dim_content this run — numeric-only feature vector stands.")

feature_frame.head()


Categorical candidates found on dim_content: ['content_type', 'competition_level', 'main_intent', 'model_used']
content_type: 0 missing (0.0%) -> filled 'unknown'
competition_level: 62,963 missing (19.6%) -> filled 'unknown'
main_intent: 60,836 missing (18.9%) -> filled 'unknown'
model_used: 84,522 missing (26.3%) -> filled 'unknown'
Feature frame with categoricals: 321,546 rows x 24 columns


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,avg_position_prev30,active_days_prev30,ctr_prev30,content_type_comparison article,content_type_feedly article,content_type_keyword article,...,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_unknown,model_used_gemini-2.5-flash,model_used_gemini-3-flash-preview,model_used_gpt-4o-mini,model_used_gpt-5-mini,model_used_unknown
0,client_3ffa76342f366962,content_5067dafdab76f97c,0.0,0.0,NaN,0,0.0,False,True,False,...,False,False,False,False,True,False,False,True,False,False
1,client_3ffa76342f366962,content_cfd5f8764543bfee,1.0,0.0,5.0,1,0.0,False,True,False,...,False,False,False,False,True,False,False,True,False,False
2,client_3ffa76342f366962,content_6d26e2f1a8a6512b,0.0,0.0,NaN,0,0.0,False,True,False,...,False,False,False,False,True,False,False,True,False,False
3,client_3ffa76342f366962,content_ab514679d7b70992,0.0,0.0,NaN,0,0.0,False,True,False,...,False,False,False,False,True,False,False,True,False,False
4,client_3ffa76342f366962,content_f1a36b8d0b297a35,0.0,0.0,NaN,0,0.0,False,True,False,...,False,False,False,False,True,False,False,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [5]:
feature_notes = pd.DataFrame([
    {"feature": "imp_prev30", "meaning": "Feb (prev30) GSC impressions, summed",
     "missing": "None — SUM of an always-present column, 0 when no impressions that period",
     "categorical": False, "available_when": "Fully observed the moment Feb ends, before March begins"},
    {"feature": "clk_prev30", "meaning": "Feb (prev30) GSC clicks, summed",
     "missing": "None — same trailing SUM",
     "categorical": False, "available_when": "Same trailing count, nothing from March"},
    {"feature": "avg_position_prev30", "meaning": "Feb average GSC position, real positions only (0 = no position data excluded via NULLIF)",
     "missing": f"NaN for the {n_unranked:,} items with zero real position data all of prev30 — kept as NaN, not zero-filled",
     "categorical": False, "available_when": "Computed entirely from Feb's daily rows"},
    {"feature": "ctr_prev30", "meaning": "Feb clicks / Feb impressions",
     "missing": "Filled 0 when imp_prev30 == 0 (no opportunity yet, not a missing measurement)",
     "categorical": False, "available_when": "A ratio of two prev30 counts only"},
    {"feature": "active_days_prev30", "meaning": "Count of Feb days with any impressions",
     "missing": "None — a tally of past days only",
     "categorical": False, "available_when": "No future information, past days only"},
])

# Append one row per categorical feature Section 1 actually found — names aren't known until
# the DESCRIBE runs, so this is built dynamically rather than hardcoded.
for col in cat_features:
    feature_notes.loc[len(feature_notes)] = {
        "feature": col,
        "meaning": f"dim_content.{col} (content metadata, discovered live in Section 1)",
        "missing": "Filled 'unknown' as its own explicit category, then one-hot encoded",
        "categorical": True,
        "available_when": "Content metadata set at (or soon after) publish time — knowable well before any prev30 or March window",
    }

print(f"Feature notes: {len(feature_notes)} features")
feature_notes


Feature notes: 9 features


,feature,meaning,missing,categorical,available_when
0,imp_prev30,"Feb (prev30) GSC impressions, summed","None — SUM of an always-present column, 0 when...",False,"Fully observed the moment Feb ends, before Mar..."
1,clk_prev30,"Feb (prev30) GSC clicks, summed",None — same trailing SUM,False,"Same trailing count, nothing from March"
2,avg_position_prev30,"Feb average GSC position, real positions only ...","NaN for the 168,485 items with zero real posit...",False,Computed entirely from Feb's daily rows
3,ctr_prev30,Feb clicks / Feb impressions,Filled 0 when imp_prev30 == 0 (no opportunity ...,False,A ratio of two prev30 counts only
4,active_days_prev30,Count of Feb days with any impressions,None — a tally of past days only,False,"No future information, past days only"
5,content_type,"dim_content.content_type (content metadata, di...","Filled 'unknown' as its own explicit category,...",True,Content metadata set at (or soon after) publis...
6,competition_level,dim_content.competition_level (content metadat...,"Filled 'unknown' as its own explicit category,...",True,Content metadata set at (or soon after) publis...
7,main_intent,"dim_content.main_intent (content metadata, dis...","Filled 'unknown' as its own explicit category,...",True,Content metadata set at (or soon after) publis...
8,model_used,"dim_content.model_used (content metadata, disc...","Filled 'unknown' as its own explicit category,...",True,Content metadata set at (or soon after) publis...


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [6]:
# --- Leakage test 1: label-derived features ---
# Timeline: prev30 = [Jan 30, Mar 1) is used for every feature above; March = [Mar 1, Apr 1) is
# the label window. They do not overlap.
march = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_march
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END_EXCL}'
    GROUP BY 1, 2
""").df()

data = feature_frame.merge(march, on=["client_hash_id", "content_hash_id"], how="inner")
data = data[data["imp_prev30"] > 0].copy()
data["is_declining"] = (data["imp_march"] < 0.8 * data["imp_prev30"]).astype(int)

base_rate = data["is_declining"].mean()
print(f"Base rate (is_declining == 1): {base_rate:.3f} — every AUC below sits next to this number.")

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

feature_cols = [c for c in feature_frame.columns if c not in ("client_hash_id", "content_hash_id")]

def quick_auc(cols, df):
    X = df[cols].fillna(0)
    y = df["is_declining"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

honest_auc = quick_auc(feature_cols, data)
print(f"Honest AUC (full feature vector — numeric + categorical, no March data): {honest_auc:.3f}")

# The trap: re-add the label's own source data as a fake feature.
data["imp_march_leak"] = data["imp_march"]
leaky_auc = quick_auc(feature_cols + ["imp_march_leak"], data)
print(f"Leaky AUC (+ raw March impressions as a feature): {leaky_auc:.3f}")

data = data.drop(columns=["imp_march_leak"])
restored_auc = quick_auc(feature_cols, data)
print(f"Honest AUC restored (leak column deleted): {restored_auc:.3f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Base rate (is_declining == 1): 0.285 — every AUC below sits next to this number.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Honest AUC (full feature vector — numeric + categorical, no March data): 0.661
Leaky AUC (+ raw March impressions as a feature): 1.000
Honest AUC restored (leak column deleted): 0.661


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [7]:
# --- Leakage test 2: future/overlapping windows ---
print("Timeline check:")
print(f"  features summed/averaged over prev30 : [{PREV30_START}, {PREV30_END_EXCL})")
print(f"  label (is_declining) computed over    : [{MONTH_START}, {MONTH_END_EXCL})")
print("No feature query above touches report_date >= MONTH_START. dim_content metadata used for")
print("categoricals is a static content-level table (not report_date-partitioned), so it carries")
print("no future-window risk by construction.")

# --- Leakage test 3: product/decision flags ---
# Re-scan both dim tables for anything that smells like an existing score, flag, or priority
# system output — a broader net than the EXCLUDE_LIKE filter Section 1 used to pick categoricals.
FLAG_LIKE = ("score", "flag", "priority", "risk", "rank", "declin", "trend", "action", "recommend", "status")
for tname in ("dim_content", "dim_clients"):
    schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES[tname]}").df()
    hits = [c for c in schema["column_name"] if any(bad in c.lower() for bad in FLAG_LIKE)]
    print(f"{tname}: flag-like columns found = {hits or 'none'} — none of these are in feature_cols above.")


Timeline check:
  features summed/averaged over prev30 : [2026-01-30, 2026-03-01)
  label (is_declining) computed over    : [2026-03-01, 2026-04-01)
No feature query above touches report_date >= MONTH_START. dim_content metadata used for
categoricals is a static content-level table (not report_date-partitioned), so it carries
no future-window risk by construction.
dim_content: flag-like columns found = none — none of these are in feature_cols above.
dim_clients: flag-like columns found = none — none of these are in feature_cols above.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [8]:
EXCLUDED_FIELDS = {
    "access_profile": "FlyRank account/commercial metadata (dim_clients) — describes the client relationship, not observed search performance.",
    "client_hash_id / content_hash_id": "Pseudonymous identifiers — grouping/join keys only, never features (memorizing an ID isn't learning a signal).",
    "gsc_data_start / ga4_data_start": "Per-client history-start dates (dim_clients) — used only to reason about data availability, not as a per-page signal.",
    "imp_march / imp_march_leak": "Raw March impressions — label-derived by construction (Leakage test 1 above); this IS is_declining's own source data.",
    "fact_content_query_90d (content_visible_query_count, rare_impressions_share, anonymized_impressions_share)":
        "Not used this round — this table's fixed 90-day window isn't confirmed to sit strictly before MONTH_START the way fact_daily is explicitly filtered, so including it without that timing check risks the same future-window leak as Leakage test 2. Flagged for a dedicated check before promotion to a feature.",
}

# Anything Leakage test 3 flagged as score/flag/priority-like also gets excluded here, even if it
# turned out empty this run — kept dynamic so the list stays honest against whatever the live
# schema actually contained, not a guess made ahead of time.
for tname in ("dim_content", "dim_clients"):
    schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES[tname]}").df()
    hits = [c for c in schema["column_name"] if any(bad in c.lower() for bad in FLAG_LIKE)]
    for h in hits:
        EXCLUDED_FIELDS[f"{tname}.{h}"] = "Flag/score/priority-like column (Leakage test 3) — a product decision-flag would encode an old rule, not the world; never used as a feature."

print(f"Excluded fields ({len(EXCLUDED_FIELDS)}):")
for field, reason in EXCLUDED_FIELDS.items():
    print(f"- {field}: {reason}")


Excluded fields (5):
- access_profile: FlyRank account/commercial metadata (dim_clients) — describes the client relationship, not observed search performance.
- client_hash_id / content_hash_id: Pseudonymous identifiers — grouping/join keys only, never features (memorizing an ID isn't learning a signal).
- gsc_data_start / ga4_data_start: Per-client history-start dates (dim_clients) — used only to reason about data availability, not as a per-page signal.
- imp_march / imp_march_leak: Raw March impressions — label-derived by construction (Leakage test 1 above); this IS is_declining's own source data.
- fact_content_query_90d (content_visible_query_count, rare_impressions_share, anonymized_impressions_share): Not used this round — this table's fixed 90-day window isn't confirmed to sit strictly before MONTH_START the way fact_daily is explicitly filtered, so including it without that timing check risks the same future-window leak as Leakage test 2. Flagged for a dedicated check before pr

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Addendum — client-grouped split check (not part of the ML-05 card)

*Diagnostic, not a rubric section: the Leakage-hunt AUCs above (Section 3) used a random 70/30
split. `skills/hunting-leakage-and-validating/SKILL.md` warns a random split can let a model
memorize per-client quirks — a real risk with only ~55 clients in this slice. This checks
whether 0.664 survives a client-grouped split, and adds Precision@50 (this repo's actual
primary metric — `CLAUDE.md`, `scripts/ml_utils.py`), before any baseline/model notebook gets
built on top of these numbers.*

In [9]:
# Does the AUC above survive a client-grouped split, and what does Precision@50 (this repo's
# actual primary metric — CLAUDE.md, scripts/ml_utils.py) look like? Checked before ML-07/08 are
# built on top of the random-split number above.
from sklearn.model_selection import GroupShuffleSplit

X_full = data[feature_cols].fillna(0)
y_full = data["is_declining"]
groups = data["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X_full, y_full, groups=groups))

n_train_clients = groups.iloc[train_idx].nunique()
n_test_clients = groups.iloc[test_idx].nunique()
overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"Client-grouped split: {n_train_clients} train clients, {n_test_clients} test clients, "
      f"overlap={len(overlap)} (must be 0 for an honest holdout)")

model_grouped = LogisticRegression(max_iter=2000).fit(X_full.iloc[train_idx], y_full.iloc[train_idx])
test_scores = model_grouped.predict_proba(X_full.iloc[test_idx])[:, 1]
test_labels = y_full.iloc[test_idx]

grouped_auc = roc_auc_score(test_labels, test_scores)
print(f"\nRandom-split AUC (Section 3 above): {honest_auc:.3f}")
print(f"Client-grouped AUC:                 {grouped_auc:.3f}")
print(f"Gap (random - grouped): {honest_auc - grouped_auc:+.3f} — per skills/hunting-leakage-and-validating: "
      "\"if you can't explain the gap, you're not done.\"")

# Precision@50 — this repo's actual primary metric (CLAUDE.md, scripts/ml_utils.py
# precision_at_k), not yet computed anywhere in ML-04/05/06. Ranks the client-grouped test set
# by predicted probability and checks what fraction of the top 50 are truly declining.
ranked = pd.DataFrame({"score": test_scores, "label": test_labels.to_numpy()}).sort_values("score", ascending=False)
precision_at_50 = ranked.head(50)["label"].mean()
test_base_rate = test_labels.mean()
print(f"\nPrecision@50 (client-grouped test set, n_test={len(test_idx):,}): {precision_at_50:.3f}")
print(f"Test-set base rate (what a random top-50 would score): {test_base_rate:.3f}")
if test_base_rate > 0:
    print(f"Lift over base rate: {precision_at_50 / test_base_rate:.2f}x")


Client-grouped split: 29 train clients, 13 test clients, overlap=0 (must be 0 for an honest holdout)

Random-split AUC (Section 3 above): 0.661
Client-grouped AUC:                 0.629
Gap (random - grouped): +0.032 — per skills/hunting-leakage-and-validating: "if you can't explain the gap, you're not done."

Precision@50 (client-grouped test set, n_test=75,733): 0.440
Test-set base rate (what a random top-50 would score): 0.320
Lift over base rate: 1.38x


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
